In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## PULSE — Wikipedia EventStreams 수신 + AI 규칙 생성
# MAGIC - Wikipedia SSE → Bronze 적재
# MAGIC - GPT-4o 규칙 생성 (최초 1회)
# MAGIC - GX 품질 검사

# COMMAND ----------
import os
import sys
import json
import requests
from sseclient import SSEClient
from datetime import datetime

# Key Vault 설정 ← 프로젝트에 맞게 수정
os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"

sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
print("[OK] Key Vault 연결 완료")

# COMMAND ----------
# ADLS 연결
storage_client = vault.get_storage_client("datacopsadls")
print("[OK] ADLS 연결 완료")

# Azure OpenAI 키 가져오기
openai_key      = vault.get_secret("azure-openai-key")
openai_endpoint = vault.get_secret("azure-openai-endpoint")
print("[OK] Azure OpenAI 연결 완료")

In [0]:
# COMMAND ----------
import json
import requests
import pandas as pd
from sseclient import SSEClient
from datetime import datetime

WIKI_STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"
headers = {
    "Accept": "text/event-stream",
    "User-Agent": "datacops-data-quality/0.1"
}

def collect_events(stream_url, max_count=1000, filters=None):
    """
    범용 SSE 이벤트 수집 함수
    stream_url: SSE 스트림 URL
    max_count:  수집할 건수
    filters:    dict 형태의 필터 조건 (예: {"wiki": "kowiki"})
    """
    response = requests.get(stream_url, stream=True, headers=headers)
    client = SSEClient(response)
    events = []

    for event in client.events():
        if event.event != "message":
            continue
        try:
            data = json.loads(event.data)
        except json.JSONDecodeError:
            continue

        # 필터 적용 (없으면 전체 수집)
        if filters:
            if not all(data.get(k) == v for k, v in filters.items()):
                continue

        events.append(data)

        if len(events) >= max_count:
            break

    return events

# 전체 Wikipedia 이벤트 1000건 수집 (범용 — 필터 없음)
print("Wikipedia 이벤트 수집 시작...")
raw_events = collect_events(
    stream_url=WIKI_STREAM_URL,
    max_count=1000
)
print(f"[OK] {len(raw_events)}건 수집 완료")

In [0]:
# COMMAND ----------
def auto_profile(data: list[dict]) -> dict:
    """
    어떤 데이터든 받아서 칼럼 정보 자동 분석
    범용 설계 — 데이터 구조에 의존하지 않음
    """
    df = pd.json_normalize(data)

    profile = {}
    for col in df.columns:
        series = df[col].dropna()

        # 칼럼 타입 감지
        if series.empty:
            dtype = "unknown"
        elif pd.api.types.is_bool_dtype(series):
            dtype = "boolean"
        elif pd.api.types.is_numeric_dtype(series):
            dtype = "numeric"
        else:
            # 날짜 감지 시도
            try:
                pd.to_datetime(series.head(10), unit='s')
                dtype = "timestamp"
            except Exception:
                # 카테고리 vs 문자열 구분
                unique_ratio = series.nunique() / len(series)
                dtype = "categorical" if unique_ratio < 0.05 else "string"

        col_info = {
            "dtype":       dtype,
            "null_rate":   round(df[col].isna().mean(), 3),
            "unique_count": int(series.nunique()),
            "sample":      series.head(5).tolist()
        }

        # 숫자형은 통계 추가
        if dtype == "numeric":
            col_info.update({
                "min": float(series.min()),
                "max": float(series.max()),
                "mean": round(float(series.mean()), 3)
            })

        profile[col] = col_info

    return profile

# 프로파일링 실행
profile = auto_profile(raw_events)
print(f"[OK] 칼럼 {len(profile)}개 분석 완료")
for col, info in list(profile.items())[:5]:
    print(f"  {col}: {info['dtype']} (null={info['null_rate']}, unique={info['unique_count']})")